# Example 7.10.1 / 1: Order Identical Bins

Companion notebook for **7.10.1 Symmetry Breaking** in the AMPL lab Documentation, section "Order Identical Bins". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Order Identical Bins

For identical bins indexed in a chosen order, prefer lower-numbered bins: a bin may be used only
if the previous one is. Six items of sizes 6, 5, 4, 4, 3 and 2 go into bins of capacity 10:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                        # items to pack
set BINS ordered;                 # identical bins; the order sets which to use first
param size {ITEMS} > 0;           # size of each item
param capacity > 0;               # capacity of every bin
var Assign {ITEMS, BINS} binary;  # 1 if the item goes into the bin
var Use {BINS} binary;            # 1 if the bin is used

minimize BinsUsed: sum {b in BINS} Use[b];
subject to PlaceOnce {i in ITEMS}: sum {b in BINS} Assign[i,b] = 1;  # each item packed once
subject to BinCapacity {b in BINS}:                                    # only used bins hold items
    sum {i in ITEMS} size[i] * Assign[i,b] <= capacity * Use[b];
# symmetry breaking: a bin is used only if the previous bin is used
subject to UseInOrder {b in BINS: ord(b) > 1}:
    Use[b] <= Use[prev(b)];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b", "c", "d", "e", "f"]
ampl.param["size"] = {"a": 6, "b": 5, "c": 4, "d": 4, "e": 3, "f": 2}
ampl.param["capacity"] = 10
ampl.set["BINS"] = [1, 2, 3, 4, 5]

In [5]:
# run.py
ampl.var["Use"][1].fix(0)  # ask for a plan that leaves bin 1 empty
for rows in ["without UseInOrder", "with UseInOrder"]:
    if rows.startswith("without"):
        ampl.con["UseInOrder"].drop()
    else:
        ampl.con["UseInOrder"].restore()
    ampl.solve(solver="highs", verbose=False)
    print(rows, "|", ampl.solve_result, ampl.obj["BinsUsed"].value(), ampl.var["Use"].to_dict())

without UseInOrder | solved 3.0 {1: 0, 2: 1, 3: 1, 4: 1, 5: 0}
with UseInOrder | infeasible 0.0 {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}


### Pure AMPL: data.dat and model.run

The items need three bins. Without the ordering rows, a plan that uses bins 2, 3 and 4 is just as
good as one that uses 1, 2 and 3, and the solver may explore both. With `UseInOrder`, leaving bin
1 empty is infeasible: only plans whose used bins form a prefix remain. Without the `fix`, both
versions find three bins, so the ordering removed duplicates and kept the optimum.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := a b c d e f;
param size := a 6  b 5  c 4  d 4  e 3  f 2;
param capacity := 10;
set BINS := 1 2 3 4 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
option solver_msg 0;
fix Use[1] := 0;
drop UseInOrder;
solve;
display solve_result, Use;
restore UseInOrder;
solve;
display solve_result;
""")
except Exception:
    pass  # AMPL has printed the error above

solve_result = solved

Use [*] :=
1  0
2  1
3  1
4  1
5  0
;



[WARNING] 
	presolve, constraint PlaceOnce['f']:
		all variables eliminated, but lower bound = 1 > 0
	presolve, constraint PlaceOnce['e']:
		all variables eliminated, but lower bound = 1 > 0
	presolve, constraint PlaceOnce['d']:
		all variables eliminated, but lower bound = 1 > 0
	presolve, constraint PlaceOnce['c']:
		all variables eliminated, but lower bound = 1 > 0
	presolve, constraint PlaceOnce['b']:
		all variables eliminated, but lower bound = 1 > 0
	1 presolve messages suppressed.


solve_result = infeasible

